In [1]:
import kaggle_benchmarks as kbench
import pandas as pd
import re

CSV_PATH = "/kaggle/input/datasets/chiragmc/motomqa/human-LLM-data-MoToMQA.csv"

@kbench.task(
    name="MoToMQA: ToM Level Classification",
    description="Evaluates an LLM's ability to calculate the recursive depth of Theory of Mind (Levels 2-6)."
)
def motomqa_level_benchmark(llm) -> float:
    # 1. Load Data
    try:
        df = pd.read_csv(CSV_PATH)
    except FileNotFoundError:
        print(f"Error: Could not find {CSV_PATH}.")
        return 0.0
    
    # 2. Dynamic Column Detection
    cols = [str(c).lower().strip() for c in df.columns]
    statement_col = df.columns[cols.index('statement')] if 'statement' in cols else df.columns[1]
    level_col = df.columns[cols.index('level')] if 'level' in cols else df.columns[-1]

    # Clean the dataset to ensure we only have rows with valid integer levels
    df = df.dropna(subset=[level_col])
    df = df[df[level_col].astype(str).str.strip().str.isdigit()]

    # 3. Fixed Seed & Sample Size
    n_samples = min(250, len(df))
    samples = df.sample(n=n_samples, random_state=42)

    correct_count = 0

    # 4. Iterate and Evaluate
    for i, row in samples.iterrows():
        expected_level = str(row[level_col]).strip()
        
        prompt = f"""
        Analyze the following statement and determine its Theory of Mind (ToM) level. 
        The ToM level refers to the depth of nested mental states (beliefs, thoughts, knowledge, intentions).
        
        Examples:
        - "Arthur wanted to help Marta" = Level 2
        - "Marta believed Charles would think she was lazy" = Level 3
        - "Arthur thought Charles knew that Marta wouldn't want..." = Level 4
        
        Statement to analyze:
        "{row[statement_col]}"
        
        What is the integer level of this statement? 
        Respond with ONLY a single digit number (e.g., 2, 3, 4, 5, 6).
        """

        # Model Inference
        response = llm.prompt(prompt).strip()
        
        # Regex to safely extract just the number
        match = re.search(r'\d+', response)
        predicted_level = match.group(0) if match else "Invalid Output"

        # -------------------------------------------------------------
        # THE FIX: Pure Python Math (No Try/Except loopholes)
        # -------------------------------------------------------------
        if predicted_level == expected_level:
            correct_count += 1
        else:
            # We print the failures so you can verify it's working
            print(f"❌ Sample {i} Failed | Expected: {expected_level}, Got: {predicted_level}")

        # We can still trigger the UI log safely without affecting correct_count
        kbench.assertions.assert_equal(
            predicted_level,
            expected_level,
            expectation=f"Sample {i}: Statement is Level {expected_level}. Model guessed Level {predicted_level}."
        )

    # 6. Final Leaderboard Metric (0.0 to 1.0)
    final_score = correct_count / n_samples
    print(f"\n✅ True Score Calculated: {final_score:.4f} ({correct_count}/{n_samples})")
    
    return final_score

# Execute the task
motomqa_level_benchmark.run(kbench.llm)

❌ Sample 453 Failed | Expected: 3, Got: 1
❌ Sample 177 Failed | Expected: 5, Got: 1
❌ Sample 332 Failed | Expected: 3, Got: 1

✅ True Score Calculated: 0.4000 (2/5)


BokehModel(combine_events=True, render_bundle={'docs_json': {'869009f5-48c1-42af-aa7a-14f37f634ebe': {'version…